# Temporal rescoring v2: camera motion, centre-distance linking, gap filling, box smoothing

Version 1 (`temporal.py`, see `temporal_rescoring.ipynb`) links same-class boxes in consecutive frames by IoU and pulls each box's score towards its track's best scores. Two weaknesses on drone video:
- **The camera moves.** A small object that shifts a few pixels between frames has little or no IoU with itself, so it is not linked — and small objects are most of the test set.
- **It only re-ranks.** A box the detector missed in one frame stays missing.

Version 2 (`temporal.temporal_v2`) adds four switches, each compared alone against version 1, then combined:

| Switch | What it does | Source |
|---|---|---|
| `motion` | Estimate a homography between consecutive frames (Lucas-Kanade optical flow on corner points + RANSAC) and move the previous boxes into the current frame before matching | camera-motion compensation |
| `center_dist` | Also link boxes whose centres are within `center_dist` × box size (and similar area), not only IoU ≥ 0.5 | REPP (Sabater et al., 2020) |
| `max_gap` + `fill` | A track may skip up to `max_gap` frames; the missing boxes are inserted (moved by the camera motion, interpolated) with score `fill_decay` × the neighbours' | box propagation |
| `smooth_sigma` | Gaussian smoothing of each track's box coordinates over frames | REPP |

Two inputs: **Faster R-CNN baseline** (multi-scale, the detections behind the 0.2182 → 0.2263 result of version 1) and the **best ensemble** from `ensemble_temporal.ipynb`.

**Run after** `ensemble_temporal.ipynb` and `ensemble_weights.ipynb` have finished (it needs `runs/ensemble/`). CPU only, about 30–40 min with the ensemble input only (`RUN_INPUTS`); nothing here uses the GPU. Outputs: `runs/temporal/results_v2.csv` and test submissions in `runs/temporal/submissions/`.

## Setup

In [1]:
import json
import pickle
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

SRC = Path.cwd().parent            # this notebook lives in src/experiments/
ROOT = SRC.parent
sys.path.insert(0, str(SRC))
from common.evaluate import evaluate_detections
from common.pipeline import fuse
from common.submission import write_submission
import temporal as T

DATA_ROOT = ROOT / "multiple-object-video-detection-hw-2-ucu-ai-2026"
GT_FILE = ROOT / "data_processed" / "val" / "gt_eval_coco.json"
TEST_CSV = DATA_ROOT / "test_images.csv"
RUNS = ROOT / "runs"
ENS = RUNS / "ensemble"
OUT = RUNS / "temporal"
(OUT / "submissions").mkdir(parents=True, exist_ok=True)

val_meta = json.loads((DATA_ROOT / "val" / "annotations_coco.json").read_text())["images"]
VAL = [{"image_id": im["id"], "video": im["video"], "frame": im["frame_id"],
        "path": DATA_ROOT / "val" / im["file_name"], "shape": (im["height"], im["width"])} for im in val_meta]
test = pd.read_csv(TEST_CSV)
TEST = [{"image_id": k, "video": c, "frame": int(f), "path": DATA_ROOT / p, "shape": (h, w)}
        for k, c, f, p, w, h in zip(test["image_id"], test["clip"], test["frame"], test["file_name"], test["width"], test["height"])]
LABELLED = sorted(im["id"] for im in json.loads(GT_FILE.read_text())["images"])
labelled = set(LABELLED)
FRAME_IDS = {"val": {(f["video"], f["frame"]): f["image_id"] for f in VAL},
             "test": {(f["video"], f["frame"]): f["image_id"] for f in TEST}}
COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"]
print(f"val: {len(VAL)} frames ({len(LABELLED)} labelled) | test: {len(TEST)} frames")

val: 2846 frames (359 labelled) | test: 3811 frames


## 1. Camera motion

One homography per pair of consecutive frames, cached in `runs/temporal/homographies_<split>.pkl`. The table shows how far the camera moves the centre of the frame between consecutive frames, per clip: motion compensation can only matter where this is several pixels.

In [2]:
def homographies(split, frames):
    path = OUT / f"homographies_{split}.pkl"
    if path.exists():
        return pickle.loads(path.read_bytes())
    t0 = time.time()
    H = T.estimate_homographies(frames)
    path.write_bytes(pickle.dumps(H))
    print(f"{split}: {len(H)} homographies in {(time.time() - t0) / 60:.1f} min")
    return H


def motion_table(H, frames):
    shape = {(f["video"], f["frame"]): f["shape"] for f in frames}
    rows = []
    for (video, fr), h in H.items():
        hh, ww = shape[(video, fr)]
        c = np.array([ww / 2, hh / 2, 1.0])
        p = h @ c
        rows.append({"clip": video, "shift_px": float(np.linalg.norm(p[:2] / p[2] - c[:2])),
                     "identity": bool(np.allclose(h, np.eye(3)))})
    t = pd.DataFrame(rows)
    return t.groupby("clip").agg(frames=("shift_px", "size"), median_shift_px=("shift_px", "median"),
                                 p90_shift_px=("shift_px", lambda s: s.quantile(0.9)), failed=("identity", "mean")).round(2)


H_VAL = homographies("val", VAL)
motion_val = motion_table(H_VAL, VAL)
motion_val

,frames,median_shift_px,p90_shift_px,failed
clip,,,,
uav0000086_00000_v,463,0.54,1.12,0.0
uav0000117_02622_v,348,3.53,13.42,0.0
uav0000137_00458_v,232,2.11,21.01,0.0
uav0000182_00000_v,362,1.88,12.79,0.0
uav0000268_05773_v,977,1.02,2.19,0.0
uav0000305_00000_v,183,0.51,23.90,0.0
uav0000339_00001_v,274,0.77,11.08,0.0


## 2. Inputs

- **Faster R-CNN:** `runs/faster_rcnn/baseline/inference/ms_valall_dets.json` (from `infer.py --split valall --scales 720:1280,1080:1920 --val-res test --tag ms`).
- **Ensemble:** the best weighted fusion from `ensemble_weights.ipynb` (`runs/ensemble/fused_best_*.pkl`) if that notebook was run; otherwise the best equal-weight configuration in `runs/ensemble/results.csv`, fused here on every val frame (cached in `runs/temporal/`).

In [3]:
final_b = json.loads((RUNS / "pipeline" / "final_config.json").read_text())
B_MEMBERS = list(final_b["members"])
OURS = ["frcnn_baseline_ms", "retinanet_baseline_ms"]
SETS = {"A: B's 7": B_MEMBERS, "B: B's 7 + Faster R-CNN": B_MEMBERS + OURS[:1],
        "B: B's 7 + RetinaNet": B_MEMBERS + OURS[1:], "B: B's 7 + Faster R-CNN + RetinaNet": B_MEMBERS + OURS}

ens_results = pd.read_csv(ENS / "results.csv")
best_fused = ens_results[ens_results["kind"] == "fused"].sort_values("mAP", ascending=False).iloc[0]
ENS_NAMES, ENS_IOU = SETS[best_fused["config"]], float(best_fused["wbf_iou"])
if all((ENS / f"fused_best_{s}.pkl").exists() for s in ("valall", "test")):
    print("ensemble: best weighted fusion from ensemble_weights.ipynb:", json.loads((ENS / "weights_best.json").read_text())["label"])
else:
    print(f"ensemble: {best_fused['config']}, WBF IoU {ENS_IOU}: val mAP {best_fused['mAP']:.4f}")


def fused_dets(split, frames):
    best = ENS / f"fused_best_{'valall' if split == 'val' else 'test'}.pkl"   # from ensemble_weights.ipynb (QUICK = False)
    if all((ENS / f"fused_best_{s}.pkl").exists() for s in ("valall", "test")):   # only if val and test are the same fusion
        return pickle.loads(best.read_bytes())
    path = OUT / f"fused_{split}.pkl"
    if path.exists():
        return pickle.loads(path.read_bytes())
    preds = {n: pickle.loads((ENS / f"{'valall' if split == 'val' else 'test'}_{n}.pkl").read_bytes()) for n in ENS_NAMES}
    t0 = time.time()
    dets = []
    for f in frames:
        xyxy, s, c = fuse([preds[n][f["image_id"]] for n in ENS_NAMES], f["shape"], iou=ENS_IOU)
        dets += [{"image_id": f["image_id"], "video": f["video"], "frame": f["frame"], "category_id": int(k) + 1,
                  "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)], "score": float(v)}
                 for (x1, y1, x2, y2), v, k in zip(xyxy, s, c)]
    path.write_bytes(pickle.dumps(dets))
    print(f"fused {len(frames)} {split} frames in {(time.time() - t0) / 60:.1f} min")
    return dets


INPUTS = {
    "Faster R-CNN": lambda: json.loads((RUNS / "faster_rcnn/baseline/inference/ms_valall_dets.json").read_text()),
    "Ensemble": lambda: fused_dets("val", VAL),
}
RUN_INPUTS = ["Ensemble"]   # add "Faster R-CNN" to also compare on a single detector (about +30 min)
INPUTS = {k: v for k, v in INPUTS.items() if k in RUN_INPUTS}
# Version 1's best setting per input: from temporal.py's grid (Faster R-CNN) and ensemble_temporal.ipynb (ensemble)
tr = pd.read_csv(RUNS / "temporal_results.csv")
tr = tr[(tr["run"] == "baseline") & (tr["tag"] == "ms") & (tr["setting"] == "rescore")].sort_values("mAP", ascending=False).iloc[0]
et = ens_results[ens_results["kind"] == "temporal"].sort_values("mAP", ascending=False).iloc[0]
V1 = {name: dict(link_iou=float(r["link_iou"]), lam=float(r["lam"]), top_k=int(r["top_k"]), single_penalty=float(r["single_penalty"]))
      for name, r in [("Faster R-CNN", tr), ("Ensemble", et)]}
V1

ensemble: best weighted fusion from ensemble_weights.ipynb: B7+Ret+RT1280+Y26m | size p=2 + FRCNN vote 0.5 +add


{'Faster R-CNN': {'link_iou': 0.5,
  'lam': 0.7,
  'top_k': 15,
  'single_penalty': 1.0},
 'Ensemble': {'link_iou': 0.5, 'lam': 0.5, 'top_k': 15, 'single_penalty': 1.0}}

## 3. One switch at a time, then combined

Each row starts from version 1's best setting for that input and turns on one switch. `combined` turns on every switch that beat version 1, each at its best value. All rows are scored on the 359 labelled val frames; the other frames only provide temporal context.

In [4]:
VARIANTS = {
    "v1": {},
    "motion": dict(motion=True),
    "center 0.5": dict(center_dist=0.5),
    "center 1.0": dict(center_dist=1.0),
    "motion + center 1.0": dict(motion=True, center_dist=1.0),
    "gap 1 + fill": dict(max_gap=1, fill=True),
    "gap 2 + fill": dict(max_gap=2, fill=True),
    "gap 2 + fill (decay 0.3)": dict(max_gap=2, fill=True, fill_decay=0.3),
    "smooth 0.6": dict(smooth_sigma=0.6),
    "smooth 1.0": dict(smooth_sigma=1.0),
}
GROUPS = [("motion",), ("center 0.5", "center 1.0"), ("gap 1 + fill", "gap 2 + fill", "gap 2 + fill (decay 0.3)"), ("smooth 0.6", "smooth 1.0")]
rows, best_cfg = [], {}


def run(name, dets, label, cfg):
    t0 = time.time()
    out = T.temporal_v2(dets, homs=H_VAL, frame_ids=FRAME_IDS["val"], **V1[name], **cfg)
    m = evaluate_detections(GT_FILE, [d for d in out if d["image_id"] in labelled], image_ids=LABELLED)
    n_fill = sum(1 for d in out if d.get("filled") and d["image_id"] in labelled)
    rows.append({"input": name, "variant": label, **m, "filled_boxes": n_fill, "seconds": round(time.time() - t0)})
    print(f"{name:13s} {label:28s} mAP {m['mAP']:.4f}  AR {m['AR']:.4f}  ({time.time() - t0:.0f} s)", flush=True)
    return m


for name, load in INPUTS.items():
    dets = load()
    base = evaluate_detections(GT_FILE, [d for d in dets if d["image_id"] in labelled], image_ids=LABELLED)
    rows.append({"input": name, "variant": "no temporal", **base})
    res = {label: run(name, dets, label, cfg) for label, cfg in VARIANTS.items()}
    # combined: in each group, the best variant if it beat v1
    combo = {}
    for group in GROUPS:
        best = max(group, key=lambda g: res[g]["mAP"])
        if res[best]["mAP"] > res["v1"]["mAP"]:
            combo.update(VARIANTS[best])
    if combo:
        res["combined"] = run(name, dets, "combined: " + ", ".join(f"{k}={v}" for k, v in combo.items()), combo)
    best_label = max(res, key=lambda k: res[k]["mAP"])
    best_cfg[name] = combo if best_label == "combined" else VARIANTS[best_label]
    del dets

table = pd.DataFrame(rows)
table.to_csv(OUT / "results_v2.csv", index=False)
show = table.set_index(["input", "variant"])[COLS + ["filled_boxes"]]
show.assign(delta_vs_v1=show["mAP"] - show.groupby(level=0)["mAP"].transform(lambda s: s.xs("v1", level=1).iloc[0])).round(4)

Ensemble      v1                           mAP 0.3191  AR 0.5250  (33 s)
Ensemble      motion                       mAP 0.3187  AR 0.5250  (42 s)
Ensemble      center 0.5                   mAP 0.3186  AR 0.5250  (33 s)
Ensemble      center 1.0                   mAP 0.3182  AR 0.5250  (33 s)
Ensemble      motion + center 1.0          mAP 0.3183  AR 0.5250  (45 s)
Ensemble      gap 1 + fill                 mAP 0.3183  AR 0.5257  (33 s)
Ensemble      gap 2 + fill                 mAP 0.3183  AR 0.5264  (33 s)
Ensemble      gap 2 + fill (decay 0.3)     mAP 0.3184  AR 0.5264  (34 s)
Ensemble      smooth 0.6                   mAP 0.3200  AR 0.5258  (33 s)
Ensemble      smooth 1.0                   mAP 0.3210  AR 0.5262  (32 s)
Ensemble      combined: smooth_sigma=1.0   mAP 0.3210  AR 0.5262  (32 s)


mAP    AP50    AP75  AP_small  \
input    variant                                                        
Ensemble no temporal                 0.3138  0.5722  0.3052    0.1525   
         v1                          0.3191  0.5809  0.3097    0.1837   
         motion                      0.3187  0.5816  0.3087    0.1921   
         center 0.5                  0.3186  0.5804  0.3089    0.1905   
         center 1.0                  0.3182  0.5799  0.3083    0.1911   
         motion + center 1.0         0.3183  0.5809  0.3084    0.1920   
         gap 1 + fill                0.3183  0.5791  0.3091    0.1845   
         gap 2 + fill                0.3183  0.5790  0.3094    0.1841   
         gap 2 + fill (decay 0.3)    0.3184  0.5791  0.3094    0.1841   
         smooth 0.6                  0.3200  0.5817  0.3096    0.1864   
         smooth 1.0                  0.3210  0.5819  0.3093    0.1896   
         combined: smooth_sigma=1.0  0.3210  0.5819  0.3093    0.1896   

                                     AP_medium  AP_large      AR  \
input    variant                                                   
Ensemble no temporal                    0.2682    0.5127  0.5250   
         v1                             0.2731    0.5200  0.5250   
         motion                         0.2728    0.5177  0.5250   
         center 0.5                     0.2725    0.5205  0.5250   
         center 1.0                     0.2718    0.5188  0.5250   
         motion + center 1.0            0.2724    0.5168  0.5250   
         gap 1 + fill                   0.2724    0.5187  0.5257   
         gap 2 + fill                   0.2725    0.5193  0.5264   
         gap 2 + fill (decay 0.3)       0.2725    0.5192  0.5264   
         smooth 0.6                     0.2745    0.5206  0.5258   
         smooth 1.0                     0.2741    0.5211  0.5262   
         combined: smooth_sigma=1.0     0.2741    0.5211  0.5262   

                                     filled_boxes  delta_vs_v1  
input    variant                                                
Ensemble no temporal                          NaN      -0.0053  
         v1                                   0.0       0.0000  
         motion                               0.0      -0.0005  
         center 0.5                           0.0      -0.0006  
         center 1.0                           0.0      -0.0009  
         motion + center 1.0                  0.0      -0.0008  
         gap 1 + fill                      2883.0      -0.0009  
         gap 2 + fill                      5580.0      -0.0008  
         gap 2 + fill (decay 0.3)          5580.0      -0.0008  
         smooth 0.6                           0.0       0.0008  
         smooth 1.0                           0.0       0.0019  
         combined: smooth_sigma=1.0           0.0       0.0019

**Observations:** _fill in_

## 4. Per clip: does motion compensation help more where the camera moves more?

Version 1 vs the best version 2 setting, per val clip, next to the clip's camera motion from section 1.

In [5]:
clip_of = {f["image_id"]: f["video"] for f in VAL}
clips = sorted(motion_val.index)
per_clip = []
for name, load in INPUTS.items():
    dets = load()
    outs = {"v1": T.temporal_v2(dets, homs=H_VAL, frame_ids=FRAME_IDS["val"], **V1[name]),
            "best v2": T.temporal_v2(dets, homs=H_VAL, frame_ids=FRAME_IDS["val"], **V1[name], **best_cfg[name])}
    for clip in clips:
        ids = [i for i in LABELLED if clip_of[i] == clip]
        row = {"input": name, "clip": clip, "median_shift_px": motion_val.loc[clip, "median_shift_px"]}
        for k, out in outs.items():
            row[k] = evaluate_detections(GT_FILE, [d for d in out if d["image_id"] in ids], image_ids=ids)["mAP"]
        per_clip.append(row)
    del dets, outs
per_clip = pd.DataFrame(per_clip).assign(delta=lambda t: t["best v2"] - t["v1"])
print("best version 2 settings:", best_cfg)
per_clip.round(4)

best version 2 settings: {'Ensemble': {'smooth_sigma': 1.0}}


,input,clip,median_shift_px,v1,best v2,delta
0,Ensemble,uav0000086_00000_v,0.54,0.3207,0.3292,0.0084
1,Ensemble,uav0000117_02622_v,3.53,0.3279,0.3290,0.0012
2,Ensemble,uav0000137_00458_v,2.11,0.3598,0.3557,-0.0042
3,Ensemble,uav0000182_00000_v,1.88,0.2515,0.2531,0.0015
4,Ensemble,uav0000268_05773_v,1.02,0.2059,0.2069,0.0010
5,Ensemble,uav0000305_00000_v,0.51,0.4240,0.4232,-0.0008
6,Ensemble,uav0000339_00001_v,0.77,0.3284,0.3282,-0.0003


**Observations:** _fill in_

## 5. Test submissions

The best version 2 setting per input, applied to the test frames:
`runs/temporal/submissions/ensemble_temporal_v1.csv` (version 1 on the same fusion), `ensemble_temporal_v2.csv`, and `frcnn_temporal_v2.csv` if Faster R-CNN is in `RUN_INPUTS`. The ensemble one needs the test predictions from `ensemble_temporal.ipynb` section 5.

In [6]:
H_TEST = homographies("test", TEST)
print(motion_table(H_TEST, TEST)["median_shift_px"].describe().round(2))


def write(dets, name, cfg, v1):
    out = T.temporal_v2(dets, homs=H_TEST, frame_ids=FRAME_IDS["test"], **v1, **cfg)
    preds = {}
    for d in out:
        x, y, w, h = d["bbox"]
        preds.setdefault(d["image_id"], []).append((d["category_id"] - 1, d["score"], x, y, w, h))
    sub = write_submission(preds, TEST_CSV, OUT / "submissions" / name)
    print(f"wrote {name}: {len(sub)} rows, {sum(1 for d in out if d.get('filled'))} filled boxes")


if "Faster R-CNN" in INPUTS:
    dets = json.loads((RUNS / "faster_rcnn/baseline/inference/ms_test_dets.json").read_text())
    write(dets, "frcnn_temporal_v2.csv", best_cfg["Faster R-CNN"], V1["Faster R-CNN"])
    del dets
test_dets = fused_dets("test", TEST)
write(test_dets, "ensemble_temporal_v1.csv", {}, V1["Ensemble"])          # version 1 on the same fusion, for comparison
write(test_dets, "ensemble_temporal_v2.csv", best_cfg["Ensemble"], V1["Ensemble"])

test: 3763 homographies in 0.5 min
count    48.00
mean      3.44
std       3.60
min       0.28
25%       0.90
50%       2.38
75%       4.63
max      21.07
Name: median_shift_px, dtype: float64
wrote ensemble_temporal_v1.csv: 3811 rows, 0 filled boxes
wrote ensemble_temporal_v2.csv: 3811 rows, 0 filled boxes
